# Setup

In [1]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()
from helper_functions import (
    load_episodes, 
    calculate_rating_metrics,
    calculate_imdb_metrics,
    merge_metrics_dicts
)
import statsmodels.formula.api as stats

In [2]:
# Config
GEMINI_MODEL = "gemini-2.5-flash-lite"
DEBERTA_MODEL = "microsoft_deberta_v3_base_SGD_without_dropout_cls_results"
ROOT = Path(os.environ.get("PROJECT_ROOT"))
DATA_DIR = ROOT / "data"
IMDb_RATINGS_PATH = DATA_DIR / "imdb_episode_ratings.json"
OUTPUT_DIR = DATA_DIR / "modeled_episode_ratings"
DATA_SOURCES = ["labels", "gemini", "deberta"]
DATA_PATHS = {
    "labels": 
        DATA_DIR / "labeled_scenes.json",
    "deberta": 
        DATA_DIR / "DeBERTa_predictions" / "all_scenes_with_transcripts" / f"{DEBERTA_MODEL}.json",
    "gemini":
        DATA_DIR / "gemini_predictions" /  "all_scenes_with_transcripts" / f"{GEMINI_MODEL}.json",
}

# Calculate metrics using helper functions

In [3]:
# Calculate metrics using helper functions
episodes = load_episodes()
rating_metrics = calculate_rating_metrics(episodes)
imdb_metrics = calculate_imdb_metrics()
merge_metrics_dicts(rating_metrics, imdb_metrics)

# Convert merge metric dictionaries for each source into dataframes
metrics_dfs = []
for source in DATA_SOURCES:
    metrics_dfs.append(pd.json_normalize(rating_metrics[source].values()))

# Unpack metrics dataframes so we can analyze them separately
labels_metrics_df, gemini_metrics_df, deberta_metrics_df = metrics_dfs

# Analysis

## Labels
#### Starting by looking at the episodes I manually labeled

In [4]:
labels_metrics_df

,funny_mean,funny_var,funny_change,sad_mean,sad_var,sad_change,episode_id,imdb_rating_mean,imdb_rating_variance,imdb_rating_share_1,imdb_rating_share_10
0,2.020000,0.958776,0.816327,1.580000,0.575102,0.448980,1_01,8.306875,2.932065,0.022584,0.245433
1,2.173077,1.126320,0.803922,1.865385,1.216817,0.666667,3_14,9.371446,3.632755,0.037713,0.816821
2,2.156250,0.781250,0.806452,2.125000,1.532258,0.967742,5_20,9.177798,4.183887,0.040028,0.741913
3,1.731707,0.651220,0.875000,2.024390,1.824390,0.400000,1_04,8.996665,2.854143,0.024560,0.516070
4,2.200000,0.929412,0.970588,1.685714,0.692437,0.558824,2_18,8.391839,3.016799,0.026554,0.248705
5,2.368421,0.725462,0.756757,1.500000,0.527027,0.405405,4_14,8.142857,3.368442,0.027875,0.226481
6,2.545455,0.926407,1.238095,1.818182,0.632035,0.761905,6_05,7.637333,3.518251,0.033778,0.149333
7,1.911111,0.946465,0.750000,2.066667,1.063636,0.772727,1_08,8.032447,2.632458,0.022872,0.162234
8,2.750000,0.861111,1.074074,1.428571,0.253968,0.407407,4_07,7.733878,3.431220,0.033469,0.141224
9,2.709677,1.146237,0.966667,1.580645,0.851613,0.566667,5_08,8.051012,3.612782,0.035628,0.190283


#### We can see here that we only have 15 data points since I only labeled 15 episodes manually. There is some variation in our metrics. As a reminder, those are:

- Funny/Sad Mean: The mean rating of scenes in the episode
- Funny/Sad Variance: The rating variance among scenes in the episode
- Funny/Sad Change: The mean absolute value change in funny/value between scenes, measuring the oscillation in tone in the show

- IMDB Rating Mean: The mean rating of the episode by users on IMDB
- IMDB Rating Variance: The variance of episode ratings from users on IMDB
- IMDB Share 1/10: The share of ratings that are 1/10, each measuring extreme like/dislike of the episode by users on IMDB

#### Since we only have 15 data points, including all these features in a regression wouldn't give us a realistic model, so I'll just show correlations here:

In [5]:
# Correlations
labels_metrics_df.drop(columns=["episode_id"]).corr()

,funny_mean,funny_var,funny_change,sad_mean,sad_var,sad_change,imdb_rating_mean,imdb_rating_variance,imdb_rating_share_1,imdb_rating_share_10
funny_mean,1.000000,0.215088,0.540956,-0.547356,-0.604593,-0.053846,-0.563731,0.476696,0.546150,-0.432608
funny_var,0.215088,1.000000,0.345746,-0.156704,0.001333,-0.085572,0.025670,-0.172229,-0.088742,0.022432
funny_change,0.540956,0.345746,1.000000,-0.417797,-0.365835,-0.227030,-0.483972,0.018353,0.138673,-0.358063
sad_mean,-0.547356,-0.156704,-0.417797,1.000000,0.821556,0.755440,0.531915,0.072685,0.022193,0.478064
sad_var,-0.604593,0.001333,-0.365835,0.821556,1.000000,0.436923,0.767410,0.043925,-0.007594,0.691950
sad_change,-0.053846,-0.085572,-0.227030,0.755440,0.436923,1.000000,0.243224,0.459907,0.441086,0.301661
imdb_rating_mean,-0.563731,0.025670,-0.483972,0.531915,0.767410,0.243224,1.000000,0.181802,0.105562,0.954352
imdb_rating_variance,0.476696,-0.172229,0.018353,0.072685,0.043925,0.459907,0.181802,1.000000,0.922897,0.370464
imdb_rating_share_1,0.546150,-0.088742,0.138673,0.022193,-0.007594,0.441086,0.105562,0.922897,1.000000,0.321885
imdb_rating_share_10,-0.432608,0.022432,-0.358063,0.478064,0.691950,0.301661,0.954352,0.370464,0.321885,1.000000


Some themes are evident here. Among the features derived from my labels, the variance in sadness feature has the strongest correlation with both the IMDb mean and the share of users who gave a perfect rating to the episode. This helps illustrate that Scrubs' peaks are driven by emotional depth and range, even if humor carries the show for most of the time. I would've expected that the sad_change would have a stronger correlation with IMDb mean ratings and the share of perfect ratings, but thinking back on the episodes I labeled, some of the most emotional scenes lasted for more than 150 words, which we define as the unit of a scene for labeling purposes. With that in mind, it makes sense that the variance in sadness would correlate more strongly with acclaimed episodes' ratings. IMDb rating variance and the share of 1-star ratings are pretty correlated with one another, and neither seems to have a strong relationship with the sadness variables. There is actually a measured negative correlation between the mean in funny ratings and the share of 1-star ratings, but I imagine this is due to my super-fan status, and preference for more niche, character-based humor that IMDb users who haven't seen the show a million times might not pick up on. Likewise, I might have undervalued some of the slapstick, shock-value type humor that was more common in the early 2000s and lost appeal with multiple watches. 

## Gemini Predictions
#### Moving onto the Gemini-based predictions for the 125 episodes with transcripts on the Scrubs Fandom Wiki site

In [6]:
gemini_metrics_df

,funny_mean,funny_var,funny_change,sad_mean,sad_var,sad_change,episode_id,imdb_rating_mean,imdb_rating_variance,imdb_rating_share_1,imdb_rating_share_10
0,2.791667,0.423759,0.511111,2.125000,1.473404,0.822222,1_01,8.306875,2.932065,0.022584,0.245433
1,2.803922,0.680784,0.400000,2.196078,1.320784,0.940000,3_14,9.371446,3.632755,0.037713,0.816821
2,2.677419,0.759140,0.758621,2.903226,1.823656,0.793103,5_20,9.177798,4.183887,0.040028,0.741913
3,2.700000,0.728205,0.605263,2.350000,1.720513,0.710526,1_04,8.996665,2.854143,0.024560,0.516070
4,2.969697,0.467803,0.419355,2.060606,1.308712,0.838710,2_18,8.391839,3.016799,0.026554,0.248705
...,...,...,...,...,...,...,...,...,...,...,...
120,3.263158,0.415363,0.540541,1.921053,1.371977,0.567568,6_03,7.752458,3.301547,0.029491,0.151028
121,2.954545,0.140693,0.238095,3.000000,1.142857,0.952381,6_04,8.026711,3.471741,0.032554,0.198664
122,3.136364,0.885281,0.952381,3.136364,1.742424,1.095238,6_06,8.641381,6.448165,0.063563,0.607376
123,3.181818,0.441558,0.571429,2.136364,1.170996,1.095238,6_07,7.940856,4.464984,0.045136,0.212451


#### It's worth testing adding in controls for season, whether an episode is a season opener, and whether an episode is a season finale

In [7]:
SEASON_LENGTHS = {1: 24, 2: 22, 3: 22, 4: 25, 5: 24, 6: 22} # No s7/8 transcripts

def add_controls(metrics_df: pd.DataFrame) -> pd.DataFrame:
    """
    Add season, episode number, and premiere/finale dummy controls to dataframe
    """
    
    # Parse episode_id into season and episode number
    season_episode = metrics_df["episode_id"].str.split("_", expand=True)
    metrics_df["season"] = season_episode[0].astype(int)
    metrics_df["episode_num"] = season_episode[1].astype(int)
    
    # Add premiere and finale dummy variables
    metrics_df["is_premiere"] = (metrics_df["episode_num"] == 1).astype(int)
    metrics_df["season_length"] = metrics_df["season"].map(SEASON_LENGTHS)
    metrics_df["is_finale"] = (metrics_df["episode_num"] == metrics_df["season_length"]).astype(int)
    
    # Return without intermediate season length and episode num vars
    return metrics_df.drop(columns=["season_length", "episode_num"])

In [8]:
gemini_metrics_df = add_controls(gemini_metrics_df)
gemini_metrics_df

,funny_mean,funny_var,funny_change,sad_mean,sad_var,sad_change,episode_id,imdb_rating_mean,imdb_rating_variance,imdb_rating_share_1,imdb_rating_share_10,season,is_premiere,is_finale
0,2.791667,0.423759,0.511111,2.125000,1.473404,0.822222,1_01,8.306875,2.932065,0.022584,0.245433,1,1,0
1,2.803922,0.680784,0.400000,2.196078,1.320784,0.940000,3_14,9.371446,3.632755,0.037713,0.816821,3,0,0
2,2.677419,0.759140,0.758621,2.903226,1.823656,0.793103,5_20,9.177798,4.183887,0.040028,0.741913,5,0,0
3,2.700000,0.728205,0.605263,2.350000,1.720513,0.710526,1_04,8.996665,2.854143,0.024560,0.516070,1,0,0
4,2.969697,0.467803,0.419355,2.060606,1.308712,0.838710,2_18,8.391839,3.016799,0.026554,0.248705,2,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120,3.263158,0.415363,0.540541,1.921053,1.371977,0.567568,6_03,7.752458,3.301547,0.029491,0.151028,6,0,0
121,2.954545,0.140693,0.238095,3.000000,1.142857,0.952381,6_04,8.026711,3.471741,0.032554,0.198664,6,0,0
122,3.136364,0.885281,0.952381,3.136364,1.742424,1.095238,6_06,8.641381,6.448165,0.063563,0.607376,6,0,0
123,3.181818,0.441558,0.571429,2.136364,1.170996,1.095238,6_07,7.940856,4.464984,0.045136,0.212451,6,0,0


#### Let's try different combinations of OLS models:

In [9]:
controls = "+ is_premiere + is_finale + season"
features_options = [
    "funny_mean + sad_mean", 
    "funny_var + sad_var",
    "funny_change + sad_change",
]
dependent_var_options = [
    "imdb_rating_mean",
    "imdb_rating_variance",
    "imdb_rating_share_1",
    "imdb_rating_share_10"
]

# Build different models 
for dependent_var in dependent_var_options:
    for features in features_options:
        for with_controls in [True, False]:
            controls = controls if with_controls else ""
            model = stats.ols(f"{dependent_var} ~ {features} {controls}", 
                            data=gemini_metrics_df)
            results = model.fit()
            print(results.summary())

                            OLS Regression Results                            
Dep. Variable:       imdb_rating_mean   R-squared:                       0.140
Model:                            OLS   Adj. R-squared:                  0.104
Method:                 Least Squares   F-statistic:                     3.863
Date:                Tue, 10 Mar 2026   Prob (F-statistic):            0.00280
Time:                        20:46:25   Log-Likelihood:                -29.101
No. Observations:                 125   AIC:                             70.20
Df Residuals:                     119   BIC:                             87.17
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       9.1273      0.783     11.661      

#### Let's take a closer look at the two strongest models: sadness variance predicting mean IMDb ratings, and mean funniness/sadness predicting IMDb rating variance:

In [10]:
target_models = [
    ("imdb_rating_mean", "funny_var + sad_var"),
    ("imdb_rating_variance", "funny_mean + sad_mean"),
]

for dependent_var, features in target_models:
    model = stats.ols(f"{dependent_var} ~ {features}", data=gemini_metrics_df)
    results = model.fit()
    print(results.summary())

                            OLS Regression Results                            
Dep. Variable:       imdb_rating_mean   R-squared:                       0.182
Model:                            OLS   Adj. R-squared:                  0.169
Method:                 Least Squares   F-statistic:                     13.57
Date:                Tue, 10 Mar 2026   Prob (F-statistic):           4.77e-06
Time:                        20:46:25   Log-Likelihood:                -25.948
No. Observations:                 125   AIC:                             57.90
Df Residuals:                     122   BIC:                             66.38
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      7.5447      0.104     72.296      0.0

As we saw with the data that I labeled, the regression results using our Gemini-generated ratings tell a pretty consistent story across different models. Sadness variance is the only feature that consistently appears as a predictor of IMDb mean rating and the share of IMDb ratings of 10, significant at the .01 level.  In our simple model predicting the mean IMDb rating with funny/sad variances and without controls, a one-unit increase in sadness variance is associated with a 0.35 point increase in IMDb mean rating, holding funny variance constant.

The results around IMDb rating polarization are a little more interesting. Both the funny mean and sad mean features have significant positive associations with IMDb rating variance and the share of IMDb users who gave 1-star ratings. In the simple model predicting IMDb rating variance with funny/sad means without controls, a one-unit increase in funny mean is associated with a 1.09 point increase in IMDb rating variance, and a one-unit increase in sad mean is associated with a 0.78 point increase, both significant at the .01 level. Thinking about these results, it seems that some IMDb users enjoy when episodes are more intense and/or funny, while others prefer a more balanced pace to the episode.  

Again, my oscillation variables do not perform well as predictors. Scene-to-scene funniness or sadness switching isn't significantly associated with any dependent variable. I think the variance does a better job of measuring the "gut punch" feeling, for the scene identification reasons discussed above.

Also, it's worth noting that even the best-performing models explain only about 17% or 18% of the variation in IMDb dependent variables

## DeBERTa Predictions
#### Moving onto the DeBERTa-based predictions for the 125 episodes with transcripts on the Scrubs Fandom wiki site

In [11]:
# Build different models 
for dependent_var in dependent_var_options:
    for features in features_options:
        for with_controls in [True, False]:
            controls = controls if with_controls else ""
            model = stats.ols(f"{dependent_var} ~ {features} {controls}", 
                            data=deberta_metrics_df)
            results = model.fit()
            print(results.summary())

                            OLS Regression Results                            
Dep. Variable:       imdb_rating_mean   R-squared:                       0.040
Model:                            OLS   Adj. R-squared:                  0.024
Method:                 Least Squares   F-statistic:                     2.530
Date:                Tue, 10 Mar 2026   Prob (F-statistic):             0.0838
Time:                        20:46:25   Log-Likelihood:                -35.962
No. Observations:                 125   AIC:                             77.92
Df Residuals:                     122   BIC:                             86.41
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      8.7353      0.297     29.374      0.0

Unfortunately, the results using DeBERTa predictions were pretty bleak. Most of our predictors had high p-values across all models, and the few models that did produce results significant at the .05 level came with very large standard errors, suggesting they were not meaningful. For example, in the model predicting mean IMDb rating with funny and sad variance, sad variance had a coefficient of 3.43 — nearly ten times the 0.35 we saw in the equivalent Gemini model, but with a standard error of 1.35. The results from the DeBERTa model are also largely inconsistent with the Gemini results. For example, the DeBERTa model predicting IMDb rating variance with funny and sad means shows funny mean negatively associated with rating variance, the opposite of what Gemini found. These factors, particularly the high p-values, suggest that the DeBERTa model performed more poorly than the Gemini model in correctly predicting how funny and sad scenes are.